Weapons and Ammunition data analysis


In [11]:
import pandas as pd
pd.options.display.max_columns = None

df = pd.read_csv('data/amcdata_weapons_facilities_V2.csv', encoding='latin-1')
df = df[df['summary_category'] != 1]
materials_df = df[df['item_type'] == 2]

### Select weapons life cycle stages

In [12]:
cols_to_keep = [
    # Identifiers
    'item_type', 'item',

    # Lifecycle stages - ban flags
    'ban_development',
    'ban_testing',
    'ban_production',
    'ban_acquisition',
    'ban_possession',
    'ban_station',
    'ban_transfer',
    'ban_use',
    'ban_disposal',

    # Lifecycle stages - restriction flags
    'restriction_development',
    'testing_restriction',       # note: inconsistent naming in the dataset
    'restriction_production',
    'restriction_acquisition',
    'restriction_possession',
    'restriction_transfer',
    'restriction_use',
    'restriction_disposal',

    # End-of-life stages (no ban/restriction equivalents)
    'eliminitation',             # note: typo in the dataset
    'conversion',
    'modernization',
    'facility_destruction',
]

materials_lifecycle_df = materials_df[cols_to_keep]


Which columns are causing problems due to no variance?

In [13]:
ban_cols = [c for c in materials_lifecycle_df.columns if c.startswith('ban_')]
restriction_cols = [c for c in materials_lifecycle_df.columns if 'restriction' in c]

flag_cols = ban_cols + restriction_cols

for col in flag_cols:
    unique_vals = materials_lifecycle_df[col].dropna().unique()
    if len(unique_vals) <= 1:
        print(f"{col}: only contains {unique_vals}")

ban_development: only contains [0]
ban_testing: only contains [0]
ban_production: only contains [0]
ban_possession: only contains [0]
ban_station: only contains [0]
ban_use: only contains [0]
ban_disposal: only contains [0]
restriction_production: only contains [0]
restriction_acquisition: only contains [0]
restriction_disposal: only contains [0]


### Testing correlations between different life cycle stages of weapons/ammunitions

In [15]:
# Separate ban and restriction columns
ban_cols = [c for c in materials_lifecycle_df.columns if c.startswith('ban_')]
restriction_cols = [c for c in materials_lifecycle_df.columns if 'restriction' in c]

# Correlation between every ban col vs every restriction col
corr_matrix = materials_lifecycle_df[ban_cols + restriction_cols].corr()

# Slice to only show ban vs restriction (not ban vs ban or restriction vs restriction)
corr_ban_vs_restriction = corr_matrix.loc[ban_cols, restriction_cols]
print(corr_ban_vs_restriction)

                 restriction_development  testing_restriction  \
ban_development                      NaN                  NaN   
ban_testing                          NaN                  NaN   
ban_production                       NaN                  NaN   
ban_acquisition                -0.096523             -0.13901   
ban_possession                       NaN                  NaN   
ban_station                          NaN                  NaN   
ban_transfer                   -0.096523             -0.13901   
ban_use                              NaN                  NaN   
ban_disposal                         NaN                  NaN   

                 restriction_production  restriction_acquisition  \
ban_development                     NaN                      NaN   
ban_testing                         NaN                      NaN   
ban_production                      NaN                      NaN   
ban_acquisition                     NaN                      NaN   
ban_posse

In [16]:
# Stack matrix into a series and sort
corr_ranked = (
    corr_ban_vs_restriction
    .stack()
    .reset_index()
    .rename(columns={'level_0': 'ban', 'level_1': 'restriction', 0: 'correlation'})
    .sort_values('correlation', ascending=False)
)

print(corr_ranked.head(40))

               ban              restriction  correlation
0  ban_acquisition  restriction_development    -0.096523
3  ban_acquisition     restriction_transfer    -0.096523
8     ban_transfer     restriction_transfer    -0.096523
5     ban_transfer  restriction_development    -0.096523
1  ban_acquisition      testing_restriction    -0.139010
4  ban_acquisition          restriction_use    -0.139010
9     ban_transfer          restriction_use    -0.139010
6     ban_transfer      testing_restriction    -0.139010
2  ban_acquisition   restriction_possession    -0.173494
7     ban_transfer   restriction_possession    -0.173494


In [17]:
# Pearson - default, fine for binary
materials_lifecycle_df[ban_cols + restriction_cols].corr(method='pearson')

,ban_development,ban_testing,ban_production,ban_acquisition,ban_possession,ban_station,ban_transfer,ban_use,ban_disposal,restriction_development,testing_restriction,restriction_production,restriction_acquisition,restriction_possession,restriction_transfer,restriction_use,restriction_disposal
ban_development,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ban_testing,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ban_production,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ban_acquisition,NaN,NaN,NaN,1.000000,NaN,NaN,1.000000,NaN,NaN,-0.096523,-0.139010,NaN,NaN,-0.173494,-0.096523,-0.139010,NaN
ban_possession,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ban_station,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ban_transfer,NaN,NaN,NaN,1.000000,NaN,NaN,1.000000,NaN,NaN,-0.096523,-0.139010,NaN,NaN,-0.173494,-0.096523,-0.139010,NaN
ban_use,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ban_disposal,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
restriction_development,NaN,NaN,NaN,-0.096523,NaN,NaN,-0.096523,NaN,NaN,1.000000,0.694365,NaN,NaN,-0.064194,-0.035714,-0.051434,NaN


In [18]:
# Spearman - better for ordinal/binary data, more robust
materials_lifecycle_df[ban_cols + restriction_cols].corr(method='spearman')

,ban_development,ban_testing,ban_production,ban_acquisition,ban_possession,ban_station,ban_transfer,ban_use,ban_disposal,restriction_development,testing_restriction,restriction_production,restriction_acquisition,restriction_possession,restriction_transfer,restriction_use,restriction_disposal
ban_development,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ban_testing,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ban_production,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ban_acquisition,NaN,NaN,NaN,1.000000,NaN,NaN,1.000000,NaN,NaN,-0.096523,-0.139010,NaN,NaN,-0.173494,-0.096523,-0.139010,NaN
ban_possession,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ban_station,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ban_transfer,NaN,NaN,NaN,1.000000,NaN,NaN,1.000000,NaN,NaN,-0.096523,-0.139010,NaN,NaN,-0.173494,-0.096523,-0.139010,NaN
ban_use,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ban_disposal,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
restriction_development,NaN,NaN,NaN,-0.096523,NaN,NaN,-0.096523,NaN,NaN,1.000000,0.694365,NaN,NaN,-0.064194,-0.035714,-0.051434,NaN
